##Team: KARVAN
Team members: Nazrin Malikova, Shohret Aghazada

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/malikovanazrin/blob/main/velo_roads_analysis.ipynb)

##Part 1: Dəqiq Geofəza Hesablamaları

In [75]:
!pip install geopandas pyogrio -q
import geopandas as gpd, pandas as pd, zipfile, os

In [76]:
import os, urllib.request, urllib.parse

# Download the datasets from the GitHub repo if they are not already here
BASE = "https://raw.githubusercontent.com/malikovanazrin/velo_roads_analysis/main/data/"
for f in ["movcud-zolaqlar-velo.kmz", "mpv-v-mps.-son.kmz"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(BASE + urllib.parse.quote(f), f)
        print("Downloaded:", f)

Downloaded: movcud-zolaqlar-velo.kmz
Downloaded: mpv-v-mps.-son.kmz


In [77]:
import zipfile

with zipfile.ZipFile("movcud-zolaqlar-velo.kmz") as z:
    print(z.namelist())
    z.extractall("lanes_kml")       # we are unzipping and extracting the file into a new folder

print(gpd.list_layers("lanes_kml/doc.kml"))

['doc.kml']
              name geometry_type
0    VELO YOL 2026       Unknown
1  Mövcud zolaqlar       Unknown


In [78]:
# Assigning a variable name to the layer inside the folder of "lanes_kml"
lanes = gpd.read_file("lanes_kml/doc.kml", layer="Mövcud zolaqlar")

##Data Inspection: Lanes dataset

In [79]:
# we define "inspect" as shortcut to make our code simpler, shorter and easier to use. It can be reused for other datasets such as parking.
def inspect(gdf, name):
    print("=====", name, "=====")
    print("Shape:", gdf.shape)
    print("CRS:", gdf.crs)
    print(gdf.geom_type.value_counts())
    print("Has Z:", gdf.geometry.has_z.any())
    print("Missing/empty geometry:", gdf.geometry.isna().sum(), gdf.geometry.is_empty.sum())
    display(gdf.head())

inspect(lanes, "Existing lanes")

===== Existing lanes =====
Shape: (61, 13)
CRS: EPSG:4326
LineString    61
Name: count, dtype: int64
Has Z: True
Missing/empty geometry: 0 1


,id,Name,description,timestamp,begin,end,altitudeMode,tessellate,extrude,visibility,drawOrder,icon,geometry
0,None,Filarmoniya bağı.,None,NaT,NaT,NaT,None,1,0,-1,NaN,None,"LINESTRING Z (49.83476 40.36278 0, 49.83383 40..."
1,None,İstiqlaliyyət küç.,None,NaT,NaT,NaT,None,1,0,-1,NaN,None,"LINESTRING Z (49.83122 40.36658 0, 49.83173 40..."
2,None,Azərbaycan pr.,None,NaT,NaT,NaT,None,1,0,-1,NaN,None,"LINESTRING Z (49.83489 40.36914 0, 49.834 40.3..."
3,None,Mirzəağa Əliyev küç.,None,NaT,NaT,NaT,None,1,0,-1,NaN,None,"LINESTRING Z (49.82686 40.37389 0, 49.83122 40..."
4,None,Bəxtiyar Vahabzadə küç.,None,NaT,NaT,NaT,None,1,0,-1,NaN,None,"LINESTRING Z (49.80976 40.37495 0, 49.8157 40...."


In [80]:
import shapely

coords = shapely.get_coordinates(lanes.geometry, include_z=True)   # all points as rows: x, y, z
z = coords[:, 2]                                                     # take only the z column

print("Z min:", z.min(), " Z max:", z.max())
print("Non-zero Z values:", (z != 0).sum())

Z min: 0.0  Z max: 0.0
Non-zero Z values: 0


After inspecting "Mövcud zolaqlar" layer, we noticed several observations: The layer containts 61 features, CRS of data is EPSG:4326; There is Z value, but it is all zeros; there is no missing geometry data, but one row has an empty geometry without coordinates written.



In [81]:
lanes[lanes.geometry.is_empty]

,id,Name,description,timestamp,begin,end,altitudeMode,tessellate,extrude,visibility,drawOrder,icon,geometry
35,None,8 Noyabr pr.,None,NaT,NaT,NaT,None,1,0,-1,NaN,None,LINESTRING EMPTY


In [82]:
text = open("lanes_kml/doc.kml", encoding="utf-8").read()
i = text.find("8 Noyabr")
print(text[i-200 : i+1500])

95944,0 49.88097746966656,40.3789380424904,0 49.88131286919428,40.37872346810272,0 49.8817795334131,40.37641374101688,0 
					</coordinates>
				</LineString>
			</Placemark>
			<Placemark>
				<name>8 Noyabr pr.</name>
				<styleUrl>#m_ylw-pushpin210202</styleUrl>
				<LineString>
					<tessellate>1</tessellate>
					<coordinates>
						49.9468696133296,40.36094434192624,0 49.94644327067476,40.36183856771282,0 49.94609394767593,40.36264006787339,0 49.94561375178246,40.36356017244507,0 49.94523928825823,40.36415362500204,0 49.94469032146701,40.36476853665996,0 49.94412544537081,40.36530236033646,0 49.94312709038729,40.36626953874701,0 49.94243073627403,40.36693893925953,0 49.94168474787058,40.36729537565958,0 49.94040720109222,40.36769729327084,0 49.93838281619248,40.36824471720094,0 49.93717404227458,40.36852425559186,0 49.93535633088218,40.36904854918529,0 49.93247560277102,40.36975771131399,0 49.93096746645535,40.37049338512983,0 49.93001510113574,40.37120148030076,0 49.928394646346

In [83]:
import re

print("Occurrences:", text.count("8 Noyabr"))

for m in re.finditer("8 Noyabr", text):
    start = m.start()
    print("=" * 80)
    print(text[start-100 : start+600])

Occurrences: 3
40.37641374101688,0 
					</coordinates>
				</LineString>
			</Placemark>
			<Placemark>
				<name>8 Noyabr pr.</name>
				<styleUrl>#m_ylw-pushpin210202</styleUrl>
				<LineString>
					<tessellate>1</tessellate>
					<coordinates>
						49.9468696133296,40.36094434192624,0 49.94644327067476,40.36183856771282,0 49.94609394767593,40.36264006787339,0 49.94561375178246,40.36356017244507,0 49.94523928825823,40.36415362500204,0 49.94469032146701,40.36476853665996,0 49.94412544537081,40.36530236033646,0 49.94312709038729,40.36626953874701,0 49.94243073627403,40.36693893925953,0 49.94168474787058,40.36729537565958,0 49.94040720109222,40.36769729327084,0 49.93838281619248,40.36824471720094,0 49.93
40.37899050023461,0 
					</coordinates>
				</LineString>
			</Placemark>
			<Placemark>
				<name>8 Noyabr pr.</name>
				<styleUrl>#m_ylw-pushpin210202</styleUrl>
				<LineString>
					<tessellate>1</tessellate>
					<coordinates>
						49.88181910960293,40.37940864744725,0 49.882

In [84]:
lanes = lanes[~lanes.geometry.is_empty]
print(lanes.shape)

(60, 13)


In [85]:
lanes["Name_clean"] = lanes["Name"].str.strip()                           # remove spaces at start/end
lanes["Name_clean"] = lanes["Name_clean"].str.replace(r"\s+", " ", regex=True)  # collapse double spaces
lanes["Name_clean"].value_counts().sort_index() #sorting alphabeticaly will help us to to spot variants easier

,count
Name_clean,
28 may HUB,2
28 may küç.,1
8 Noyabr pr.,2
Abdulla Şaiq küçəsi,1
Afiyəddin Cəlilov küç.,1
Akademik Zərifə Əliyeva küç.,1
Aleksandr Puşkin küç.,1
Azadlıq pr.,1
Azərbaycan pr.,1


In [86]:
!pip install folium mapclassify -q

In [87]:
check = lanes[lanes["Name_clean"].isin(["Kövkəb Xanım Səfərəliyeva küç.", "Keykab xanım Səfərəliyeva 2", "Üzeyir H - Keykab S. - keçid"])]

m = lanes.explore(color="lightgray", tiles="CartoDB positron")
check.explore(m=m, column="Name_clean", style_kwds={"weight": 5})
m

/usr/local/lib/python3.13/dist-packages/geopandas/explore.py:383: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(scale_factor="{r}")


In [88]:
check = lanes[lanes["Name_clean"].isin(["Xaqani küçəsi", "Xəqani küç.", "Xəqani Rüstəmov küç."])]

m = lanes.explore(color="lightgray", tiles="CartoDB positron")
check.explore(m=m, column="Name_clean", style_kwds={"weight": 5})
m

/usr/local/lib/python3.13/dist-packages/geopandas/explore.py:383: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(scale_factor="{r}")


In [89]:
check = lanes[lanes["Name_clean"].isin(["Bulvar", "Bulvar 2"])]

m = lanes.explore(color="lightgray", tiles="CartoDB positron")
check.explore(m=m, column="Name_clean", style_kwds={"weight": 5})
m

/usr/local/lib/python3.13/dist-packages/geopandas/explore.py:383: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(scale_factor="{r}")


In [90]:
check = lanes[lanes["Name_clean"].isin(["Ağ şəhər ərazisi.", "Ağ şəhər bulvarı ərazisi"])]

m = lanes.explore(color="lightgray", tiles="CartoDB positron")
check.explore(m=m, column="Name_clean", style_kwds={"weight": 5})
m

/usr/local/lib/python3.13/dist-packages/geopandas/explore.py:383: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(scale_factor="{r}")


In [91]:
check = lanes[lanes["Name_clean"].isin(["Fəzail Bayramov küç.", "Yusif Səfərov küç.", "Fəzail Bayramov küç. - Yusif Səfərov küç."])]

m = lanes.explore(color="lightgray", tiles="CartoDB positron")
check.explore(m=m, column="Name_clean", style_kwds={"weight": 5})
m

/usr/local/lib/python3.13/dist-packages/geopandas/explore.py:383: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(scale_factor="{r}")


After visualizing the streets with similar names, we observed that Xaqani küçəsi and Xəqani küç. are continuation of each other, it means that together they make one same street. However, Xəqani Rüstəmov küç. is a completely different street. Bulvar and Bulvar 2 are continouation of each other, therefore, we considered them one street based on both the visualizations and the names.Ağ şəhər ərazisi and Ağ şəhər bulvarı ərazisi are observed as seperate streets. Also, Fəzail Bayramov küç., Yusif Səfərov küç., Fəzail Bayramov küç. - Yusif Səfərov küç. streets are also seperate based on visualizations.

Some streets have two parallel lane lines (one per side of the road). We treat each as a separate lane and sum their lengths. Only exact duplicate geometries would be removed (found: 0).

In [92]:
fixes = {
    "Xəqani küç.": "Xaqani küçəsi",
    # add one line per decision
    "Keykab xanım Səfərəliyeva 2": "Kövkəb Xanım Səfərəliyeva küç.",
    "Üzeyir H - Keykab S. - keçid": "Kövkəb Xanım Səfərəliyeva küç.",
    "Bulvar 2": "Bulvar"
}
lanes["Name_clean"] = lanes["Name_clean"].replace(fixes)
lanes["Name_clean"].value_counts().sort_index()

,count
Name_clean,
28 may HUB,2
28 may küç.,1
8 Noyabr pr.,2
Abdulla Şaiq küçəsi,1
Afiyəddin Cəlilov küç.,1
Akademik Zərifə Əliyeva küç.,1
Aleksandr Puşkin küç.,1
Azadlıq pr.,1
Azərbaycan pr.,1


In [93]:
print(lanes.geometry.duplicated().sum())

0


##Data Inspection: Parking dataset

In [94]:
import zipfile

with zipfile.ZipFile("mpv-v-mps.-son.kmz") as z:
    print(z.namelist())
    z.extractall("parkings_kml")       # we are unzipping and extracting the file into a new folder

print(gpd.list_layers("parkings_kml/doc.kml"))

['doc.kml', 'files/mysavedplaces_closed_2_0_0_1_3_0_1.png', 'files/mysavedplaces_open_2_0_0_1_3_0_1.png']
                                      name geometry_type
0                         Temporary Places       Unknown
1    velosiped məntəqələri.11 sentyabr.kmz       Unknown
2  Velosiped yolları 2025. avqust. son.kmz       Unknown
3            Velosiped yolları 2025 yanvar       Unknown
4              Ümumi 12 iyul. SON olan.kmz       Unknown
5                                My Places       Unknown
6                         Sightseeing Tour       Unknown
7               velosiped məntəqələri yeni       Unknown


In [95]:
# Assigning variable names to the layer that we need
parkings = gpd.read_file("parkings_kml/doc.kml", layer="velosiped məntəqələri yeni")

In [96]:
inspect(parkings, "Parking Locations")

===== Parking Locations =====
Shape: (185, 13)
CRS: EPSG:4326
Point    185
Name: count, dtype: int64
Has Z: True
Missing/empty geometry: 0 0


,id,Name,description,timestamp,begin,end,altitudeMode,tessellate,extrude,visibility,drawOrder,icon,geometry
0,None,MPV-1-2,None,NaT,NaT,NaT,None,-1,0,-1,NaN,None,POINT Z (49.83122 40.36578 0)
1,None,MPV-2-2,None,NaT,NaT,NaT,None,-1,0,-1,NaN,None,POINT Z (49.83476 40.36898 0)
2,None,MPV-3-2,None,NaT,NaT,NaT,None,-1,0,-1,NaN,None,POINT Z (49.83469 40.36946 0)
3,None,MPV-4-2,None,NaT,NaT,NaT,None,-1,0,-1,NaN,None,POINT Z (49.83398 40.37107 0)
4,None,MPV-5-2,None,NaT,NaT,NaT,None,-1,0,-1,NaN,None,POINT Z (49.83261 40.37422 0)


After inspecting "velosiped məntəqələri yeni" layer, we noticed several observations: The layer containts 185 rows, CRS of data is EPSG:4326; There is Z value, but it is all zeros; there is no missing or empty geometry data.

##Preperation

Converted the CRS of the datasets to EPSG:32539 to measure the length in meters and assigned them to new variables.

In [97]:
lanes_m = lanes.to_crs("EPSG:32639")
parkings_m = parkings.to_crs("EPSG:32639")

# Q 1.1: Mövcud velosiped zolaqlarının (Mövcud zolaqlar qatı, EPSG:32639 proyeksiyası ilə) ümumi uzunluğu neçə kilometrdir (vergüldən sonra 2 rəqəm) və ən uzun velosiped zolağı olan küçə/prospekt hansıdır?
(Format: "Uzunluq km, Küçə adı", məs: 17.19, 8 Səməd Vurğun k.)


##Solution

Added a new column called length_m to lanes_m variable which shows length of each road. Displayed few examples from the data.

In [98]:
lanes_m["length_km"] = lanes_m.geometry.length/1000
lanes_m[["Name_clean", "length_km"]].head()

,Name_clean,length_km
0,Filarmoniya bağı.,0.619515
1,İstiqlaliyyət küç.,0.449987
2,Azərbaycan pr.,0.685260
3,Mirzəağa Əliyev küç.,1.501500
4,Bəxtiyar Vahabzadə küç.,0.507105


Answer 1: We summed up length of all 60 rows and converted to km.

In [99]:
total_km = lanes_m["length_km"].sum()
print("Total length of the existing bicyle roads:", round(total_km, 2), "km")

Total length of the existing bicyle roads: 57.14 km


In [100]:
by_street = (lanes_m
             .groupby("Name_clean")["length_km"]
             .sum()                                   # add up the lengths of each street's segments
             .sort_values(ascending=False))           # sorting from longest to shortest

(by_street).round(2)

,length_km
Name_clean,
8 Noyabr pr.,13.86
Bulvar,2.28
Neftçilər pr-Bulvar,2.26
Xətai pr.,2.02
28 may küç.,1.94
Mətbuat pr.,1.91
Hüseyn Cavid pr.,1.84
Rəşid Behbudov küç.,1.63
Ağ şəhər bulvarı ərazisi,1.56


The table shows that 8 Noyabr pr. has the longest bicycle roads with 13.86 km length, existing the both sides of the road.

##Final answer to Q 1.1

In [101]:
top_street = lanes_m.groupby("Name_clean")["length_km"].sum().idxmax()

print(f"{total_km:.2f}, {top_street}")

57.14, 8 Noyabr pr.


#Q 1.2: velosiped məntəqələri yeni qatındakı 185 velosiped parkinqindən neçəsi mövcud velosiped zolaqlarına 50 metr və daha yaxın məsafədə yerləşir?
(Format: Yalnız tam ədəd daxil edin).


##Solution

In [102]:
# Merge all lane lines into one geometry, so each parking needs only one distance calculation
lanes_union = lanes_m.geometry.union_all()

# Distance in metres from each parking to the nearest lane
parkings_m["dist_to_lane_m"] = parkings_m.geometry.distance(lanes_union)

# Parkings within 50 m (50 m or closer)
near = parkings_m["dist_to_lane_m"] <= 50

##Final answer to Q 1.2

In [103]:
print(near.sum())

87


# Q 1.3: Parkinq koordinatlarını araşdırdıqda Bakı mərkəzi şəhər çərçivəsindən (enlik 40.30–40.46, uzunluq 49.70–50.05) kənarda qalan neçə parkinq məntəqəsi var və adlarını qeyd edin?
(Format: "Say: Məntəqə adları", məs: 4: MPV-144-2, …)


##Solution

In [104]:

import zipfile
import xml.etree.ElementTree as ET

In [105]:

K = '{http://www.opengis.net/kml/2.2}'

In [106]:

with zipfile.ZipFile('mpv-v-mps.-son.kmz') as z:
    root = ET.fromstring(z.read('doc.kml'))

In [107]:

folder = next(
    f for f in root.iter(K+'Folder')
    if f.findtext(K+'name') == 'velosiped məntəqələri yeni'
)

In [108]:

parks = []

for pm in folder.findall(K+'Placemark'):
    lon, lat = map(
        float,
        pm.find('.//'+K+'coordinates').text.strip().split(',')[:2]
    )

    parks.append((
        pm.findtext(K+'name').strip(),
        lat,
        lon
    ))

## Final Answer to Q 1.3

In [109]:

outside = sorted(
    n for n, lat, lon in parks
    if not (40.30 <= lat <= 40.46 and 49.70 <= lon <= 50.05)
)

print(f'{len(outside)}: ' + ', '.join(outside))

4: MPV-144-2, MPV-145-2, MPV-146-2, MPV-147-2


#Part 2: Strateji Təhlil və AYNA Rəhbərliyinə Təklif

#Q 2: AYNA rəhbərliyinə bu &quot;tənha&quot; parkinqləri şəbəkəyə ən tez qoşmaq üçün növbəti
mərhələdə yeni velosiped zolağının salınmasını hansı əsas istiqamət/küçə üzrə təklif
edərdiniz və niyə? (Maksimum 99 sözlə əsaslandırın).



In [110]:
lonely = parkings_m[parkings_m["dist_to_lane_m"] > 50]
print(len(lonely),"parkinq məntəqəsi tənhadır")

98 parkinq məntəqəsi tənhadır


In [111]:
m = lanes_m.explore(color="green", tiles="Esri.WorldStreetMap", style_kwds={"weight": 4})
lonely.explore(m=m, color="red", marker_kwds={"radius": 5},
               tooltip=["Name", "dist_to_lane_m"])
m

## Final answer to Q 2

Nəticələrimiz göstərir ki, ümumilikdə 98 tənha velosiped parking məntəqəsi var. Vizualizasiyamız göstərir ki, onların ən tez formada şəbəkəyə qoşulması üçün yeni zolağı Mətbuat prospekti zolağının şimal ucundan Yasamal-Yeni Yasamal istiqamətində şimala uzadıb Vaqif prospekti zolağına birləşdirmək. Çünki, 36 məntəqə (37%) tənha parking bu ərazidə yerləşir. Bu marşrut iki mövcud zolağı birləşdirdiyi üçün təcrid olunmuş yeni hissə yaratmır, əksinə şəbəkənin iki hissəsini bir-birinə bağlayır. Beləliklə, bir yeni zolaqla tənha parkinqlərin üçdə birindən çoxu birbaşa şəbəkəyə qoşulur. Sıx yaşayış ərazisi olduğu üçün istifadəçi tələbatı da yüksək olacaq.